# Harf — Phase 1 runner: thinnest end-to-end recogniser

This notebook only **installs, configures and calls** the `harf` package. Every function it runs lives in `src/harf/` and is tested in CI (rule 18: notebooks hold no logic).

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Each run gets a row in `EXPERIMENTS.md` (docs repo): copy the versions and commit printed below into it.

## 1. Runtime record (FC-01, FC-02)

In [ ]:
import platform
import sys

import PIL
import torch

print("python       :", sys.version.split()[0])
print("platform     :", platform.platform())
print("torch        :", torch.__version__, "| cuda", torch.version.cuda)
print("pillow       :", PIL.__version__)
print("gpu available:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_properties = torch.cuda.get_device_properties(0)
    gpu_memory_gigabytes = round(gpu_properties.total_memory / 1024**3, 1)
    print("gpu          :", gpu_properties.name, f"({gpu_memory_gigabytes} GB)")

## 2. Get the code and install `harf`

In [ ]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/HuMzAHCodes/harf-urdu-ocr.git"
REPOSITORY_DIRECTORY = Path("/content/harf-urdu-ocr")

# Clone on the first run of a session; afterwards pull, so the latest push is what runs.
if REPOSITORY_DIRECTORY.exists():
    subprocess.run(["git", "-C", str(REPOSITORY_DIRECTORY), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIRECTORY)], check=True)

# Editable install: Colab keeps its own GPU build of torch, because pyproject.toml
# only asks for torch>=2.5, which the preinstalled version already satisfies.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPOSITORY_DIRECTORY)],
    check=True,
)

# An editable install is picked up when Python starts, but this kernel is already
# running, so the source folder is added directly to make `import harf` work now.
source_directory = str(REPOSITORY_DIRECTORY / "src")
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

commit_result = subprocess.run(
    ["git", "-C", str(REPOSITORY_DIRECTORY), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
)
print("code commit:", commit_result.stdout.strip())

## 3. Mount Google Drive and prepare the project folders

Data and checkpoints live on Drive, never in git (D-21, `DATA.md` storage map). Creating the folders is safe to repeat: existing folders are left untouched.

In [ ]:
import os
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/harf")
PROJECT_FOLDERS = [
    "data/raw",
    "data/manifests",
    "data/shards",
    "fonts",
    "checkpoints",
    "eval",
]
for relative_folder in PROJECT_FOLDERS:
    (DRIVE_PROJECT_ROOT / relative_folder).mkdir(parents=True, exist_ok=True)

# The same settings a laptop reads from .env; src/harf/config.py will read them.
os.environ["HARF_DATA_ROOT"] = str(DRIVE_PROJECT_ROOT / "data")
os.environ["HARF_OUTPUT_ROOT"] = str(DRIVE_PROJECT_ROOT)

for relative_folder in PROJECT_FOLDERS:
    print("ready:", DRIVE_PROJECT_ROOT / relative_folder)

## 4. Check the package imports

In [ ]:
import harf

print("harf version:", harf.__version__)

## 5. Phase 1 run

Filled in as the Phase 1 concepts are built: render synthetic lines, train the CNN + CTC model, evaluate on the fixed phase-run set, write the sample output file.

## Problems we hit and what we decided

The phase record (WORKFLOW section 9). One entry per problem, written as it happens:

```text
### N. <the problem in a few words> (D-NN if there is one)

**What happened.** <The symptom, with numbers.>
**The check.** <What was tested before acting.>
**Options.** <What could have been done.>
**Decision.** <What was done, and why.>
**Lesson.** <One sentence, if one is worth keeping.>
```